# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Load metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Keywords: {metadata.keywords}")
print(f"Date Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets and their fields. Here, we inspect the record sets, their `@id` values, and the associated fields for the dataset.

We note that you must reference record sets and fields by their `@id` per Croissant best practices.

In [ ]:
# List record sets in this dataset along with their fields and columns, referencing everything by @id

print("Available Record Sets (referenced by @id):")
record_set_ids = []

# The record sets can be accessed via metadata.recordSet, with each record set having @id and fields
for rs in getattr(metadata, 'recordSet', []):
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    rs_desc = getattr(rs, 'description', None)
    print(f"- RecordSet @id: {rs_id}")
    print(f"  Name: {rs_name}")
    print(f"  Description: {rs_desc}")
    record_set_ids.append(rs_id)
    # Print fields info for this record set
    if hasattr(rs, 'field'):
        for field in rs.field:
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', None)
            data_type = getattr(field, 'dataType', None)
            print(f"    - Field @id: {field_id}, Name: {field_name}, DataType: {data_type}")
    print()
# If there are no record sets, print a message
if not record_set_ids:
    print("No record sets found in the dataset metadata.")

## 3. Data Extraction
Load tabular data from each record set into pandas DataFrames for analysis.

**Note:** As all entities in the dataset (record sets, fields, columns, etc.) must be referenced by their `@id`, below we extract using record set `@id`s.

In [ ]:
dataframes = {}

if record_set_ids:
    for rid in record_set_ids:
        # Efficiently extract up to 5 records per record set for preview (remove head() to load all)
        print(f"Extracting up to 5 records from RecordSet @id: {rid}")
        records = list(dataset.records(record_set=rid))
        dataframes[rid] = pd.DataFrame(records)
        print(f"Columns: {dataframes[rid].columns.tolist()}")
        print(dataframes[rid].head(), '\n')
else:
    print("No record sets available to extract data from.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps.

You must select a record set and fields by their `@id`. For illustration, we select the first available record set and pick a numeric field from it for filtering and normalization.

In [ ]:
import numpy as np

if record_set_ids:
    chosen_rs_id = record_set_ids[0]
    df = dataframes[chosen_rs_id]
    print(f"Working with RecordSet @id: {chosen_rs_id}")

    # Attempt to select a numeric field by inspecting field metadata for this record set
    rs_meta = None
    for rs in getattr(metadata, 'recordSet', []):
        if getattr(rs, '@id', None) == chosen_rs_id:
            rs_meta = rs
            break
    numeric_field_id = None
    group_field_id = None

    if rs_meta and hasattr(rs_meta, 'field'):
        for field in rs_meta.field:
            # Typical Croissant data types include schema:Float or schema:Integer
            dtype = getattr(field, 'dataType', None)
            if dtype in ['schema:Float', 'schema:Integer']:
                numeric_field_id = getattr(field, '@id', None)
                break
        # Optionally find a non-numeric field for grouping
        for field in rs_meta.field:
            dtype = getattr(field, 'dataType', None)
            if dtype == 'schema:Text':
                group_field_id = getattr(field, '@id', None)
                break

    # Fallback: try to select a numeric column using pandas
    if numeric_field_id is None and len(df.select_dtypes(include=[np.number]).columns) > 0:
        numeric_field_id = df.select_dtypes(include=[np.number]).columns[0]

    if numeric_field_id and numeric_field_id in df.columns:
        # For demo purposes, pick 10th percentile as threshold
        try:
            threshold = df[numeric_field_id].quantile(0.1)
        except Exception:
            threshold = 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        print(filtered_df.head())

        # Add normalized column (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a groupable field if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped data by '{group_field_id}': (showing first 5 groups)")
            print(grouped_df.head())
        else:
            print("No suitable group field found in the record set for demonstration.")
    else:
        print("No numeric field found for EDA in the selected record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(10,6))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If both numeric and group field exist, do a boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² mass spectrometry dataset exposes record sets and a variety of numeric and textual fields, suitable for downstream protein analysis or exploratory statistical workflows.
- Through `mlcroissant`, you can easily programmatically explore and process annotated data by referencing each record set and field by their unique `@id`.
- The EDA demonstrates basic filtering and normalization, and common visualizations such as distributions and group comparisons.

Further analysis may include more advanced statistical computation, feature engineering, or visual analytics based on the scientific research goals!